# ❤️ Notebook 2 — Heart Disease Dataset EDA
**Dataset**: Cleveland Heart Disease Database  
**Source**: [Kaggle — John Smith](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset)  
**Rows**: 303 | **Features**: 13 | **Target**: `target` (0 = No Heart Disease, 1 = Heart Disease)

### Objective
Understand the heart disease dataset before training Model 3 (or Model 4 in the extended cascade).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BLUE   = "#2D6A9F"
RED    = "#C0392B"
ORANGE = "#D35400"
BG     = "#F7F9FB"
DARK   = "#1C2833"

np.random.seed(42)
print("Libraries loaded ✅")


## 1 · Load Dataset

In [ ]:
def make_heart():
    """Synthesise Cleveland Heart Disease data (303 rows, 13 features)."""
    n = 303; neg = 164; pos = 139
    df = pd.DataFrame({
        'age':       np.concatenate([np.random.normal(52,9,neg), np.random.normal(56,9,pos)]).clip(29,77).astype(int),
        'sex':       np.concatenate([np.random.choice([0,1],neg,p=[0.32,0.68]), np.random.choice([0,1],pos,p=[0.17,0.83])]),
        'cp':        np.concatenate([np.random.choice([0,1,2,3],neg,p=[0.55,0.15,0.15,0.15]),
                                     np.random.choice([0,1,2,3],pos,p=[0.15,0.20,0.30,0.35])]),
        'trestbps':  np.concatenate([np.random.normal(129,17,neg), np.random.normal(134,19,pos)]).clip(94,200),
        'chol':      np.concatenate([np.random.normal(243,51,neg), np.random.normal(251,48,pos)]).clip(126,564),
        'fbs':       np.concatenate([np.random.choice([0,1],neg,p=[0.85,0.15]), np.random.choice([0,1],pos,p=[0.82,0.18])]),
        'restecg':   np.concatenate([np.random.choice([0,1,2],neg,p=[0.50,0.48,0.02]),
                                     np.random.choice([0,1,2],pos,p=[0.37,0.58,0.05])]),
        'thalach':   np.concatenate([np.random.normal(158,19,neg), np.random.normal(141,23,pos)]).clip(71,202),
        'exang':     np.concatenate([np.random.choice([0,1],neg,p=[0.68,0.32]), np.random.choice([0,1],pos,p=[0.42,0.58])]),
        'oldpeak':   np.concatenate([np.random.exponential(0.6,neg), np.random.exponential(1.6,pos)]).clip(0,6.2),
        'slope':     np.concatenate([np.random.choice([0,1,2],neg,p=[0.07,0.58,0.35]),
                                     np.random.choice([0,1,2],pos,p=[0.20,0.42,0.38])]),
        'ca':        np.concatenate([np.random.choice([0,1,2,3],neg,p=[0.65,0.22,0.09,0.04]),
                                     np.random.choice([0,1,2,3],pos,p=[0.35,0.25,0.22,0.18])]),
        'thal':      np.concatenate([np.random.choice([1,2,3],neg,p=[0.07,0.68,0.25]),
                                     np.random.choice([1,2,3],pos,p=[0.06,0.35,0.59])]),
        'target':    np.array([0]*neg + [1]*pos)
    })
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

df = make_heart()
# df = pd.read_csv('heart.csv')   # ← use with real file

print(f"Shape: {df.shape}")
df.head()


## 2 · Data Overview

In [ ]:
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
df.describe().round(2)


## 3 · Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=BG)
counts = df['target'].value_counts().sort_index()
labels = ['No Heart Disease', 'Heart Disease']
axes[0].bar(labels, counts.values, color=[BLUE, RED], edgecolor='white', width=0.5)
for bar, v in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1, str(v),
                 ha='center', va='bottom', fontweight='bold', color=DARK)
axes[0].set_title('Class Distribution', fontweight='bold'); axes[0].set_facecolor(BG)
axes[0].spines[['top','right']].set_visible(False)
axes[1].pie(counts.values, labels=labels, colors=[BLUE, RED],
            autopct='%1.1f%%', startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportions', fontweight='bold'); axes[1].set_facecolor(BG)
plt.tight_layout(); plt.show()


## 4 · Continuous Feature Distributions

In [ ]:
cont_feats = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
col_map = {0: BLUE, 1: RED}; label_map = {0: 'No HD', 1: 'HD'}

fig, axes = plt.subplots(1, 5, figsize=(20, 4), facecolor=BG)
for ax, feat in zip(axes, cont_feats):
    for outcome, clr in col_map.items():
        ax.hist(df[df['target']==outcome][feat], bins=20, alpha=0.65,
                color=clr, label=label_map[outcome], edgecolor='white', linewidth=0.4)
    ax.set_title(feat, fontweight='bold', color=DARK); ax.legend(fontsize=7)
    ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
fig.suptitle('Continuous Features by Heart Disease Status', fontsize=13, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 5 · Categorical Feature Analysis

In [ ]:
cat_feats = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
cat_labels = {
    'sex':  {0:'Female', 1:'Male'},
    'cp':   {0:'Typical', 1:'Atypical', 2:'Non-anginal', 3:'Asympt'},
    'fbs':  {0:'FBS ≤120', 1:'FBS >120'},
    'restecg': {0:'Normal', 1:'ST abnorm', 2:'LV Hyp'},
    'exang':{0:'No', 1:'Yes'},
    'slope':{0:'Up', 1:'Flat', 2:'Down'},
    'ca':   {0:'0 vessels', 1:'1', 2:'2', 3:'3'},
    'thal': {1:'Normal', 2:'Fixed defect', 3:'Reversible'},
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor=BG)
axes = axes.flatten()

for ax, feat in zip(axes, cat_feats):
    ct = df.groupby([feat, 'target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=ax, color=[BLUE, RED], edgecolor='white', alpha=0.85, width=0.7)
    if feat in cat_labels:
        ax.set_xticklabels(list(cat_labels[feat].values()), rotation=25, fontsize=7)
    ax.set_title(feat, fontweight='bold', color=DARK); ax.set_facecolor(BG)
    ax.legend(['No HD', 'HD'], fontsize=7); ax.spines[['top','right']].set_visible(False)

fig.suptitle('Categorical Features vs Heart Disease Status', fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 6 · Correlation Matrix

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 9), facecolor=BG)
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(corr.columns))); ax.set_yticklabels(corr.columns, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(corr.values[i,j])>0.35 else DARK)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Correlation Matrix — Heart Disease Dataset', fontweight='bold', color=DARK, fontsize=13)
plt.tight_layout(); plt.show()

print("\nTop correlations with target:")
print(corr['target'].sort_values(ascending=False).to_string())


## 7 · Age vs Max Heart Rate (key inverse relationship)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
for outcome, clr in col_map.items():
    sub = df[df['target']==outcome]
    axes[0].scatter(sub['age'], sub['thalach'], c=clr, alpha=0.5, s=25, label=label_map[outcome])
    axes[1].scatter(sub['chol'], sub['oldpeak'], c=clr, alpha=0.5, s=25, label=label_map[outcome])
axes[0].set(xlabel='Age', ylabel='Max Heart Rate', title='Age vs Max Heart Rate')
axes[1].set(xlabel='Cholesterol', ylabel='ST Depression (oldpeak)', title='Cholesterol vs ST Depression')
for ax in axes:
    ax.legend(fontsize=9); ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
    ax.set_title(ax.get_title(), fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 8 · Summary

| Feature | Role | Type |
|---|---|---|
| `age` | Universal confounder | Fixed |
| `sex` | Demographic | Fixed |
| `cp` | Symptom type | Fixed |
| `trestbps` | Resting blood pressure | Partially modifiable |
| `chol` | Cholesterol | Modifiable |
| `thalach` | Max heart rate | Indicator |
| `oldpeak` | ST depression | Indicator |
| `ca` | Vessels coloured | Indicator |
| `thal` | Thalassemia type | Fixed |

**Cascade inputs received from upstream models:**
- Elevated `blood_pressure` from hypertension model
- Elevated `cholesterol` signal
